# Лабораторная работа 2. Первичный анализ временных рядов

**Цель.** Загрузить временные ряды, визуально описать динамику, выделить тренд, сезонность и автокорреляцию.

Ноутбук оформлен как самостоятельная версия учебной работы. Входные файлы, если они нужны для расчётов, лежат рядом с ноутбуком или скачиваются отдельной ячейкой.


## Ход работы

Ниже последовательно выполняются подготовка данных, построение графиков, расчёт признаков или моделей и краткая интерпретация результата. Случайные процедуры фиксируются через seed, чтобы результаты можно было повторить.


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


In [ ]:
# Импортируем необходимые библиотеки

import os
from os import path
from matplotlib import pyplot as plt
import pandas as pd

In [ ]:
# с помощью системной библиотеки OS определяем путь до файла с данными ЭКГ
dirname = os.path.abspath(os.curdir)

# загружаем данные в переменную, содержащую объект библиотеки Pandas - Dataframe
tsdf_c = pd.read_csv('calm_p.csv')
# устанавливаем индекс времени для временного ряда и сортируем по нему выборку
tsdf_c.set_index('Time').sort_index()

# аналогичным образом загружаем данные о пассажирах
passengers = pd.read_csv('passengers.csv')
# неподходящий формат данных приводим к тому, с которым Pandas может работать
passengers['Month'] = pd.to_datetime(passengers['Month'])
# также устанавливаем индекс и сортируем
df = passengers.set_index('Month').sort_index()

In [ ]:
# с помощью метода head можно посмотреть первые несколько строк набора данных в
# качестве аргумента в head можно указать целое число - количество выводимых
# строк
tsdf_c.head()

In [ ]:
# функция describe приводит описательную статистику о выборке: среднее, мат
# ожидание, моду и т.д. (если данные выборки позволяют эту статистику
# подсчитать)
tsdf_c.describe()

In [ ]:
# аналогичным образом можно просмотреть любой датафрейм и сделать о нём выводы
df.head()

In [ ]:
df.describe()

In [ ]:
# с помощью функционала matplotlib описываем особый вывод ЭКГ
def plot_assignation(axp, data, xlabel, ylabel, title1):
    axp.plot(data)
    axp.set_xlabel(xlabel)
    axp.set_ylabel(ylabel)
    axp.set_title(title1 )

In [ ]:
fig, axs = plt.subplots(2,1,figsize=(20, 15))
fig.suptitle('Обследуемый # 5')

plot_assignation(axs[0], tsdf_c["1"], 'time', 'mV','Покой. Отведение')

axs[1].plot(df["Passengers"])

In [ ]:
# импортируем функцию seasonal_decompose из statsmodels
# (то есть осуществляем декомпозицию сигнала/временного ряда)
from statsmodels.tsa.seasonal import seasonal_decompose

# задаем размер графика
from pylab import rcParams
rcParams['figure.figsize'] = 11, 9


# применяем функцию к данным о перевозках
decompose = seasonal_decompose(passengers["Passengers"], period=10)
decompose.plot()
plt.show()

In [ ]:
new_ps = decompose.trend*(decompose.seasonal+1)*decompose.resid

fig, axs = plt.subplots(figsize=(20, 15))

plt.plot(new_ps)
plt.show()

In [ ]:
# удаляем компонент тренда из временного ряда...
passengers_r = passengers["Passengers"] - decompose.trend
# ...и отрисовываем обработанный и исходный ряды
passengers_r.plot()
passengers["Passengers"].plot()

In [ ]:
import numpy as np
from scipy.signal import find_peaks # функция для поиска пиков (локальных максимумов) в сигнале

Fs = 1000  # частота дискретизации

signal = tsdf_c["2"].values

# Поиск пиков (вершин импульсов) с ограничением: между пиками должно быть не менее 500 отсчётов.
peaks, _ = find_peaks(signal, distance=500)

rr_intervals = np.diff(peaks) / Fs
bpm = 60 / np.mean(rr_intervals)

In [ ]:
# Ваш код здесь
# разложение ЭКГ по составляющим
# осуществляем декомпозицию сигнала/временного ряда
from statsmodels.tsa.seasonal import seasonal_decompose

# задаем размер графика
from pylab import rcParams
rcParams['figure.figsize'] = 11, 9

import numpy as np
from scipy.signal import find_peaks

Fs = 1000  # частота дискретизации

signal = tsdf_c["2"].values

# Поиск пиков (вершин импульсов) с ограничением: между пиками должно быть не менее 500 отсчётов.
peaks, _ = find_peaks(signal, distance=500)

rr_intervals = np.diff(peaks) / Fs
bpm = 60 / np.mean(rr_intervals)

# Расчет периода для декомпозиции
period = int((60 / bpm) * Fs)

print(f"Рассчитанный BPM: {bpm:.2f}")
print(f"Период для декомпозиции: {period} отсчетов")

# Применяем функцию к данным ЭКГ
decompose = seasonal_decompose(tsdf_c["2"], period=period, model="additive")
decompose.plot()
plt.suptitle('Декомпозиция ЭКГ сигнала', fontsize=16)
plt.tight_layout()
plt.show()

# Вывод статистики по компонентам
print("\nСтатистика компонентов:")
print(f"Тренд - среднее: {np.mean(decompose.trend[~np.isnan(decompose.trend)]):.6f}")
print(f"Сезонность - среднее: {np.mean(decompose.seasonal):.6f}")
print(f"Остатки - среднее: {np.mean(decompose.resid[~np.isnan(decompose.resid)]):.6f}")
print(f"Остатки - std: {np.std(decompose.resid[~np.isnan(decompose.resid)]):.6f}")

In [ ]:
# разложение на составляюшие белый шум

import numpy as np
white_noise = np.random.normal(0,1, 500)

decompose = seasonal_decompose(white_noise, period=10, model="additive")
decompose.plot()
plt.show()

In [ ]:
# импортируем функцию, описывающую тест Дики-Фуллера
from statsmodels.tsa.stattools import adfuller

In [ ]:
# всю теорию, описанную выше, реализуем с помощью statsmodels для проверки
# временного ряда перевозок на стационарность

alpha = 0.05
name = "Пассажиры"
ts = passengers["Passengers"]

print(f'Тест Дики-Фуллера ряда {name} :')
dftest = adfuller(ts, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

if dfoutput["p-value"] < alpha:
    print(f"Значение p меньше {alpha * 100}%. Ряд стационарный.")
else:
    print(f"Значение p больше {alpha*100}%. Ряд не стационарный.")

In [ ]:
# ВАШ КОД ЗДЕСЬ

# импортируем функцию описывающую тест Дики-Фуллера
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

# Получаем сигнал ЭКГ
ecg_signal = tsdf_c["2"]

# Функция для проведения теста и вывода результатов
def test_stationarity(timeseries, title):
    # уровень значимости
    alpha = 0.05

    print(f'Тест Дики-Фуллера для ряда: {title}')
    print()

    # тест Дики-Фуллера
    dftest = adfuller(timeseries, autolag='AIC')

    # Формируем результаты в удобном виде
    dfoutput = pd.Series(
        dftest[0:4],
        index=['Test Statistic', 'p-value', '#Lags Used', 'Number of Observations Used']
    )

    # Добавляем критические значения
    for key, value in dftest[4].items():
        dfoutput['Critical Value (%s)' % key] = value

    # Выводим результаты
    print(dfoutput)
    print()

    # Интерпретация результатов
    if dfoutput["p-value"] < alpha:
        print(f"p-value ({dfoutput['p-value']:.6f}) меньше {alpha * 100}%")
        print("Ряд является Стационарным")
        return True
    else:
        print(f"p-value ({dfoutput['p-value']:.6f}) больше {alpha * 100}%")
        print("Ряд является Нестационарным")
        return False

# Проверяем исходный сигнал
is_stationary = test_stationarity(ecg_signal, "Исходный ЭКГ сигнал")

# Если ряд нестационарный, можно попробовать сделать его стационарным
if not is_stationary:
    print()
    print("Пробуем сделать ряд стационарным с помощью дифференцирования...")
    print()

    # Берем первую разность
    ecg_diff = ecg_signal.diff().dropna()

    # Проверяем стационарность первой разности
    test_stationarity(ecg_diff, "Первая разность ЭКГ сигнала")

    # Визуализируем исходный и дифференцированный сигналы
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))

    axes[0].plot(ecg_signal[:2000])  # первые 2000 точек для наглядности
    axes[0].set_title('Исходный ЭКГ сигнал (первые 2000 отсчетов)')
    axes[0].set_xlabel('Отсчеты')
    axes[0].set_ylabel('Амплитуда')
    axes[0].grid(True)

    axes[1].plot(ecg_diff[:2000])
    axes[1].set_title('Первая разность ЭКГ сигнала (первые 2000 отсчетов)')
    axes[1].set_xlabel('Отсчеты')
    axes[1].set_ylabel('Разность')
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# указываем размер окна
window = 30

# вычисляем скользящее среднее и стандартное отклонение
rolling_mean = ts.rolling(window=window).mean()
rolling_std = ts.rolling(window=window).std()

plt.figure(figsize=(15,5))
plt.title(ts.name)
plt.plot(ts[window:], label='Реальные значения', color="black")

# отрисовываем скользящее среднее
plt.plot(rolling_mean, 'g', label='MA'+str(window),
             color="red")

# отрисовываем верхний и нижний интервалы
lower_bound = rolling_mean - (1.96 * rolling_std)
upper_bound = rolling_mean + (1.96 * rolling_std)

plt.fill_between(x=ts.index, y1=lower_bound, y2=upper_bound,
                 color='lightskyblue', alpha=0.4)
plt.legend(loc='best')
# показываем сетку на графике
plt.grid(True)
plt.show()

In [ ]:
# ваш код здесь (ЭКГ)

# Указываем размер окна для скользящего среднего
window = 300

# Получаем сигнал ЭКГ (признак "2")
ecg_signal = tsdf_c["2"]

# Вычисляем скользящее среднее и стандартное отклонение
rolling_mean = ecg_signal.rolling(window=window, center=True).mean()
rolling_std = ecg_signal.rolling(window=window, center=True).std()

# Создаем фигуру
plt.figure(figsize=(15, 8))

# Отображаем первые 10000 точек
plot_length = min(10000, len(ecg_signal))
x_values = range(plot_length)

# Отрисовываем реальные значения
plt.plot(x_values, ecg_signal[:plot_length],
         label='Реальные значения ЭКГ',
         color='black',
         alpha=0.7,
         linewidth=0.8)

# Отрисовываем скользящее среднее
plt.plot(x_values, rolling_mean[:plot_length],
         label=f'Скользящее среднее MA{window}',
         color='red',
         linewidth=1.5)

# Вычисляем верхний и нижний интервалы, используем коэффициент 1.96 для 95% доверительного интервала
lower_bound = rolling_mean - (1.96 * rolling_std)
upper_bound = rolling_mean + (1.96 * rolling_std)

# Отрисовываем доверительный интервал
plt.fill_between(x=x_values,
                 y1=lower_bound[:plot_length],
                 y2=upper_bound[:plot_length],
                 color='lightskyblue',
                 alpha=0.3,
                 label='95% доверительный интервал')

# Настройки графика
plt.title('Анализ ЭКГ сигнала: скользящее среднее и полосы Боллинджера', fontsize=14)
plt.xlabel('Отсчеты времени', fontsize=12)
plt.ylabel('Амплитуда сигнала (мВ)', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Дополнительно: статистика по сигналу
print(f"Статистика по анализируемому участку ЭКГ (первые {plot_length} отсчетов):")
print(f"Среднее значение: {ecg_signal[:plot_length].mean():.6f}")
print(f"Стандартное отклонение: {ecg_signal[:plot_length].std():.6f}")
print(f"Минимальное значение: {ecg_signal[:plot_length].min():.6f}")
print(f"Максимальное значение: {ecg_signal[:plot_length].max():.6f}")

In [ ]:
import statsmodels.tsa.api as smt

In [ ]:
ts = passengers["Passengers"]


fig = plt.figure(figsize=(12, 7))
# рисуем автокорреляционную функцию
#
# изображение отрисовывается с запаздываниями по горизонтали и корреляциями по
# вертикали
ac_plot = smt.graphics.plot_acf(ts, lags=30, alpha=0.5)

# есть также функция отрисовки частичной автокорреляции
pac_plot = smt.graphics.plot_pacf(ts, lags=30, alpha=0.5)

# Частичная автокорреляция (Partial Autocorrelation) — это краткая
# характеристика взаимосвязи между наблюдением во временном ряду и наблюдениями
# на предыдущем отрезке времени, когда влияние малой задержки устранено.
# Автокорреляция состоит как из прямой, так и из косвенной корреляции.

In [ ]:
fig = plt.figure(figsize=(20, 9))
layout = (2, 2)
ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
acf_ax = plt.subplot2grid(layout, (1, 0))
pacf_ax = plt.subplot2grid(layout, (1, 1))

ts.plot(ax=ts_ax)
ts_ax.set_title('Time Series Analysis Plots')
smt.graphics.plot_acf(ts, lags=30, ax=acf_ax, alpha=0.5)
smt.graphics.plot_pacf(ts, lags=30, ax=pacf_ax, alpha=0.5)
None

## Вывод

В работе показан полный путь от подготовки временного ряда до визуализации, расчёта признаков или построения модели. Основные результаты следует оценивать по графикам и метрикам, полученным при запуске ноутбука.
